# RebarDepthCNN — Kaggle Training Notebook

**Model:** `RebarDepthCNN` with `TemporalAttention`  
**Task:** Regression — predict top rebar mat depth (inches) from a 256-sample GPR waveform  
**Input:** `(batch, 2, 256)` — channel 0 = raw normalized waveform, channel 1 = Hilbert envelope  
**Output:** single float — depth in inches  
**Target:** MAE < 0.3 inches on a held-out bridge (B170020)

In [ ]:
%%bash
pip install readgssi --quiet

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.signal import hilbert
from readgssi.readgssi import readgssi
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.interpolate import griddata
import warnings
warnings.filterwarnings('ignore')

# Detect device and verify CUDA actually works before committing to it
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    try:
        torch.zeros(4, 2, 256, device=DEVICE)
        print(f"Device: {DEVICE} (verified working)")
    except Exception as _e:
        print(f"CUDA probe failed ({_e}), falling back to CPU")
        DEVICE = torch.device('cpu')
else:
    print(f"Device: {DEVICE}")

SEARCH_START = 50
SEARCH_END = 130
MIN_DEPTH_IN = 1.5
MAX_DEPTH_IN = 12.0
BATCH_SIZE = 512
EPOCHS = 100
PATIENCE = 15
LR = 1e-3

DATA_ROOT = Path('/kaggle/input/datasets/aidenerard/infrasense-dzt-both-bridges')
print(f"DATA_ROOT exists: {DATA_ROOT.exists()}")

In [ ]:
BRIDGES = {
    'B170020': {
        'length_ft': 204.3,
        'width_ft': 24.0,
        'epsr': 6.5,
        'ns_total': 15.0,
        'n_samples': 256,
        'data_dir': DATA_ROOT / 'B170020',
        'layout': [
            (95, 1,  610, 1802, 12.0, 1.02836, False),
            (95, 2,  617, 1806,  6.0, 1.03095, False),
            (96, 2,  741, 1930, 18.0, 1.03095, True),
            (97, 1,  437, 1623,  9.0, 1.03356, False),
            (97, 2,  443, 1645,  3.0, 1.01980, False),
            (98, 1,  505, 1703, 15.0, 1.02321, True),
            (98, 2,  498, 1690, 21.0, 1.02836, True),
        ]
    },
    'B440029': {
        'length_ft': 107.5,
        'width_ft': 34.0,
        'epsr': 6.5,
        'ns_total': 15.0,
        'n_samples': 256,
        'data_dir': DATA_ROOT / 'B440029',
        'layout': [
            (799, 1, 3535, 4163, 29.0, 1.02707, False),
            (799, 2, 3535, 4163, 23.0, 1.02707, False),
            (801, 1, 4737, 5373, 26.0, 1.01415, False),
            (801, 2, 4737, 5373, 20.0, 1.01415, False),
            (803, 1, 4835, 5462, 17.0, 1.02871, False),
            (803, 2, 4835, 5462, 11.0, 1.02871, False),
            (805, 1, 4984, 5614, 14.0, 1.02381, False),
            (805, 2, 4984, 5614,  8.0, 1.02381, False),
            (807, 2, 4016, 4643,  5.0, 1.02871, False),
        ]
    }
}


def resolve_dzt(data_dir: Path, file_num: int) -> Path:
    """Find DZT file regardless of zero-padding (095 vs 95) or space vs underscore."""
    candidates = [
        data_dir / f'WISDOT24_{file_num:03d} P_1.DZT',
        data_dir / f'WISDOT24_{file_num} P_1.DZT',
        data_dir / f'WISDOT24_{file_num:03d}_P_1.DZT',
        data_dir / f'WISDOT24_{file_num}_P_1.DZT',
    ]
    for c in candidates:
        if c.exists():
            return c
    for pat in [f'*{file_num:03d}*.DZT', f'*{file_num}*.DZT']:
        matches = list(data_dir.glob(pat))
        if matches:
            return matches[0]
    return candidates[0]

In [ ]:
def load_bridge_traces(bridge_name, bridge_cfg):
    epsr = bridge_cfg['epsr']
    ns_total = bridge_cfg['ns_total']
    n_samples = bridge_cfg['n_samples']
    ns_per_sample = ns_total / n_samples
    velocity = 0.15 / np.sqrt(epsr)
    data_dir = bridge_cfg['data_dir']
    ft_per_trace = (1.0 / 19.7) * 3.28084

    all_X, all_y, all_pos = [], [], []

    print(f"\nLoading {bridge_name}...")
    print(f"{'File':>6} {'Ch':>3} {'Traces':>8} {'Depth_mean':>12} {'Depth_std':>10}")
    print("-" * 45)

    for (file_num, ch_num, start_scan, end_scan,
         offset_ft, scale_factor, reversed_flag) in bridge_cfg['layout']:

        fpath = resolve_dzt(data_dir, file_num)
        if not fpath.exists():
            print(f"  WARNING: {fpath.name} not found, skipping")
            continue

        try:
            header, data, _ = readgssi(infile=str(fpath), plotting=False, verbose=False)
        except Exception as e:
            print(f"  ERROR reading {fpath.name}: {e}")
            continue

        arr = data[ch_num - 1].astype(np.float32)
        arr = arr[:, start_scan:end_scan]
        n_traces = arr.shape[1]

        if reversed_flag:
            arr = arr[:, ::-1].copy()

        arr = arr - arr.mean(axis=0, keepdims=True)
        max_abs = np.abs(arr).max(axis=0, keepdims=True)
        max_abs = np.where(max_abs == 0, 1.0, max_abs)
        arr = arr / max_abs

        envelope = np.abs(hilbert(arr, axis=0)).astype(np.float32)
        picks = np.argmax(envelope[SEARCH_START:SEARCH_END, :], axis=0) + SEARCH_START
        depth_m = (picks * ns_per_sample * velocity) / 2.0
        depth_inches = (depth_m * 39.3701).astype(np.float32)

        valid = (depth_inches >= MIN_DEPTH_IN) & (depth_inches <= MAX_DEPTH_IN)
        arr = arr[:, valid]
        envelope = envelope[:, valid]
        depth_inches = depth_inches[valid]
        idx_valid = np.where(valid)[0]

        along_ft = idx_valid * ft_per_trace * scale_factor
        if reversed_flag:
            along_ft = n_traces * ft_per_trace * scale_factor - along_ft

        X = np.stack([arr, envelope], axis=0).transpose(2, 0, 1)
        pos = np.stack([along_ft, np.full(len(depth_inches), offset_ft)], axis=1)

        all_X.append(X)
        all_y.append(depth_inches)
        all_pos.append(pos)

        print(f"  {file_num:>6} {ch_num:>3} {len(depth_inches):>8} "
              f"{depth_inches.mean():>11.2f}\" {depth_inches.std():>9.2f}\"")

    if not all_X:
        raise RuntimeError(f"No files loaded for {bridge_name}.")

    all_X   = np.concatenate(all_X,   axis=0)
    all_y   = np.concatenate(all_y,   axis=0)
    all_pos = np.concatenate(all_pos, axis=0)
    print(f"  {'TOTAL':>6} {'':>3} {len(all_y):>8} "
          f"{all_y.mean():>11.2f}\" {all_y.std():>9.2f}\"")
    return all_X, all_y, all_pos

In [ ]:
X_b440, y_b440, pos_b440 = load_bridge_traces('B440029', BRIDGES['B440029'])
X_b170, y_b170, pos_b170 = load_bridge_traces('B170020', BRIDGES['B170020'])

X_train, y_train = X_b440, y_b440
X_val,   y_val   = X_b170, y_b170
pos_val           = pos_b170

print(f"\nTrain: {len(y_train):,} traces (B440029)")
print(f"Val:   {len(y_val):,} traces (B170020)")
print(f"\nTrain depth range: {y_train.min():.2f}\" \u2013 {y_train.max():.2f}\"")
print(f"Val   depth range: {y_val.min():.2f}\" \u2013 {y_val.max():.2f}\"")

In [ ]:
# pin_memory only works with CUDA
_pin = DEVICE.type == 'cuda'

class RebarDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if torch.rand(1) < 0.5:
                x += torch.randn_like(x) * 0.01
            if torch.rand(1) < 0.5:
                x *= torch.FloatTensor(1).uniform_(0.95, 1.05).item()
            if torch.rand(1) < 0.5:
                shift = torch.randint(-5, 6, (1,)).item()
                x = torch.roll(x, shift, dims=-1)
                if shift > 0:
                    x[:, :shift] = 0
                elif shift < 0:
                    x[:, shift:] = 0
        return x, self.y[idx]

train_loader = DataLoader(
    RebarDataset(X_train, y_train, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=_pin)

val_loader = DataLoader(
    RebarDataset(X_val, y_val, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=_pin)

In [ ]:
IN_CHANNELS = 2
CONV_CH = [32, 64, 128]
HEAD_HIDDEN = 64
PATCH_K = 1       # no patch stacking; each trace is a single 256-sample signal
N_SAMPLES = 256   # samples per signal (from BRIDGES config)


class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Linear(channels, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x.transpose(1, 2)), dim=1)
        return (x.transpose(1, 2) * w).sum(dim=1)


class RebarDepthCNN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS):
        super().__init__()
        c1, c2, c3 = CONV_CH
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, c1, 7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(c1, c2, 5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(c2, c3, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.attn = TemporalAttention(c3)
        self.head = nn.Sequential(
            nn.Linear(c3, HEAD_HIDDEN), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(HEAD_HIDDEN, 1)
        )

    def forward(self, x):
        return self.head(self.attn(self.conv(x))).squeeze(-1)


model = RebarDepthCNN(in_channels=IN_CHANNELS).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"RebarDepthCNN — {total_params:,} parameters on {DEVICE}")

In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-6)

best_mae = float('inf')
patience_left = PATIENCE
val_maes, val_rmses = [], []

print(f"\n{'Ep':>4} {'TR_loss':>9} {'Val_loss':>9} "
      f"{'Val_MAE(in)':>12} {'Val_RMSE(in)':>13} {'Best_MAE':>10} {'LR':>10}")
print("-" * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(y_train)

    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * len(yb)
            all_preds.append(pred.cpu().numpy())
            all_targets.append(yb.cpu().numpy())
    val_loss /= len(y_val)
    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    mae  = np.abs(all_preds - all_targets).mean()
    rmse = np.sqrt(((all_preds - all_targets) ** 2).mean())

    val_maes.append(mae)
    val_rmses.append(rmse)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    if mae < best_mae:
        best_mae = mae
        patience_left = PATIENCE
        torch.save(model.state_dict(), '/kaggle/working/rebar_model.pth')
        marker = '*'
    else:
        patience_left -= 1
        marker = ''

    print(f"{epoch:>4}  {tr_loss:>9.4f}  {val_loss:>9.4f}  "
          f"{mae:>11.3f}\"  {rmse:>12.3f}\"  {best_mae:>9.3f}\"  "
          f"{lr_now:>9.2e} {marker}")

    if patience_left == 0:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest model \u2192 /kaggle/working/rebar_model.pth")
print(f"Best Val MAE: {best_mae:.3f}\"  Target: < 0.300\"")
print(f"Status: {'\u2713 TARGET MET' if best_mae < 0.3 else '\u2717 NEEDS IMPROVEMENT'}")

In [ ]:
import json
from pathlib import Path as _Path

config = {
    "in_channels": IN_CHANNELS,
    "conv_channels": CONV_CH,
    "head_hidden": HEAD_HIDDEN,
    "patch_k": PATCH_K,
    "n_samples": N_SAMPLES,
    "threshold": float(best_mae),  # best validation MAE (regression model — no classification threshold)
}
config_path = _Path('/kaggle/working/rebar_model_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")
print(json.dumps(config, indent=2))
print(f"\nUpload both /kaggle/working/rebar_model.pth and {config_path} to Google Drive")

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/rebar_model.pth', map_location=DEVICE))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        pred = model(xb.to(DEVICE))
        all_preds.append(pred.cpu().numpy())
        all_targets.append(yb.numpy())
all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
residuals   = all_preds - all_targets

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax = axes[0, 0]
ax.plot(val_maes, 'b-', label='Val MAE (inches)')
ax.plot(val_rmses, 'r-', label='Val RMSE (inches)')
ax.axhline(0.3, color='g', linestyle='--', label='MAE target (0.3")')
ax.set_xlabel('Epoch'); ax.set_ylabel('Error (inches)')
ax.set_title('Training Curves \u2014 B440029\u2192B170020')
ax.legend(); ax.grid(True)

ax = axes[0, 1]
offsets = pos_val[:, 1]
unique_offsets = np.unique(offsets)
colors = cm.tab10(np.linspace(0, 1, len(unique_offsets)))
for offset, color in zip(unique_offsets, colors):
    mask = offsets == offset
    ax.scatter(all_targets[mask], all_preds[mask], c=[color], s=1, alpha=0.3, label=f'{offset:.0f} ft')
mn, mx = min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())
ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1, label='Perfect')
ax.set_xlabel('Hilbert Pick Depth (inches)'); ax.set_ylabel('Predicted (inches)')
ax.set_title(f'Predicted vs Actual \u2014 MAE={np.abs(residuals).mean():.3f}"')
ax.legend(markerscale=5, fontsize=7); ax.grid(True)

ax = axes[1, 0]
ax.hist(residuals, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(residuals.mean(), color='red', label=f'Mean={residuals.mean():.3f}"')
ax.axvline(residuals.mean() + residuals.std(), color='orange',
           linestyle='--', label=f'\u00b11\u03c3={residuals.std():.3f}"')
ax.axvline(residuals.mean() - residuals.std(), color='orange', linestyle='--')
ax.set_xlabel('Residual (inches)'); ax.set_ylabel('Count')
ax.set_title('Residual Distribution'); ax.legend(); ax.grid(True)

ax = axes[1, 1]
ax.axis('off')
rows = [['Offset (ft)', 'N traces', 'MAE (in)', 'RMSE (in)']]
for offset in unique_offsets:
    mask = offsets == offset
    rows.append([f'{offset:.0f}', f'{mask.sum():,}',
                 f'{np.abs(residuals[mask]).mean():.3f}',
                 f'{np.sqrt((residuals[mask]**2).mean()):.3f}'])
rows.append(['TOTAL', f'{len(residuals):,}',
             f'{np.abs(residuals).mean():.3f}',
             f'{np.sqrt((residuals**2).mean()):.3f}'])
tbl = ax.table(cellText=rows[1:], colLabels=rows[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
ax.set_title('Per-Lane Validation Metrics (B170020)', pad=20)

plt.suptitle('RebarDepthCNN \u2014 Trained B440029, Validated B170020', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cfg = BRIDGES['B170020']
fig, axes = plt.subplots(1, 2, figsize=(20, 5))

for ax, values, title in zip(axes,
    [all_targets, all_preds],
    ['Ground Truth (Hilbert Picks)', 'Model Predictions']):
    grid_x, grid_y = np.mgrid[0:cfg['length_ft']:800j, 0:cfg['width_ft']:100j]
    grid_d = griddata((pos_val[:, 0], pos_val[:, 1]), values, (grid_x, grid_y), method='linear')
    vmin, vmax = np.nanpercentile(all_targets, 2), np.nanpercentile(all_targets, 98)
    im = ax.pcolormesh(grid_x, grid_y, grid_d, cmap='jet', vmin=vmin, vmax=vmax, shading='auto')
    plt.colorbar(im, ax=ax, label='Rebar Depth (inches)')
    ax.set_xlabel('Along Bridge (ft)'); ax.set_ylabel('Offset from Curb (ft)')
    ax.set_title(f'B170020 \u2014 {title}')
    ax.set_xlim(0, cfg['length_ft']); ax.set_ylim(0, cfg['width_ft'])

plt.suptitle(f'Rebar Depth Map \u2014 MAE={np.abs(residuals).mean():.3f}"', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/depth_map_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Val MAE:   {np.abs(residuals).mean():.3f} inches  (target < 0.300)")
print(f"Val RMSE:  {np.sqrt((residuals**2).mean()):.3f} inches  (target < 0.500)")
print(f"Max error: {np.abs(residuals).max():.3f} inches")
print(f"Status: {'\u2713 TARGET MET' if np.abs(residuals).mean() < 0.3 else '\u2717 NEEDS IMPROVEMENT'}")
print("=" * 60)